
PROJECT: Rule-Based AI Chatbot (DecodeLabs – Project 1)
================================================================================

This script implements a smart, efficient, and personalised rule-based chatbot.
It follows the project requirements:

1. Uses a continuous loop (while True) to keep the conversation alive.
2. Handles greetings, small talk, help, and exit commands.
3. Employs a dictionary (hash map) for O(1) exact matching – avoiding long if-elif chains.
4. Includes keyword-based matching for more natural, flexible interactions.
5. Sanitises user input (lowercase, strip, remove punctuation).
6. Provides a fallback response when no rule matches.
7. Runs until the user types an exit command (bye, exit, quit, goodbye).

ADDITIONAL FEATURES (beyond basic requirements):
- Asks for the user's name at the start and remembers it.
- Inserts the user's name into every response (personalisation).
- Expanded set of intents: greetings, bot identity, user identity, time, thanks,
  yes/no, small talk (weather, love, python, food, etc.).
- Randomised replies from a list for each intent (makes the bot less repetitive).
- Clean exit with a personalised goodbye message.

Author: Noor R. Saad

Date: 2/5/2026

**SECTION 1: IMPORTS**

In [10]:
import random
import string

**SECTION 2: CLASS DEFINITION – PersonalisedChatbot**

In [11]:
# Encapsulates all chatbot logic: data (responses, user name) and methods.
class PersonalisedChatbot:

    # SECTION 2.1: INITIALISER – builds the knowledge base
   def __init__(self):
       """
        Initialises the chatbot's knowledge base:
        - exact_responses : dict for full-string matching
        - keyword_responses: dict for keyword-in-string matching
        - exit_words       : set of commands that end the chat
        - user_name        : placeholder for the user's name (set later)
        """
        # ----- PART A: EXACT MATCH RESPONSES -----
        # Keys are full commands/sentences (after sanitisation).
        # Values are LISTS of possible replies – one is chosen randomly.
        # Each reply can include {name} which will be replaced with the user's name.

       self.exact_responses = {
            # Greetings
            "hello": [
                "Hello {name}! Nice to see you.",
                "Hi {name}! How can I help you today?",
                "Greetings, {name}!"
            ],
            "hi": [
                "Hi {name}!",
                "Hey {name}, welcome back!",
                "Hello {name}, good to talk to you."
            ],
            "hey": [
                "Hey {name} :)",
                "Yo {name}! What's up?"
            ],

            # How are you
            "how are you": [
                "I'm just a program, but I'm doing great! How about you, {name}?",
                "All systems operational, {name}! Thanks for asking.",
                "Doing well, {name}. And you?"
            ],
            "how are you doing": [
                "I'm functioning perfectly, {name}. Thanks!",
                "Great, {name}! Ready to chat."
            ],

            # Bot's identity
            "what is your name": [
                "I'm ChatBot from DecodeLabs, {name}. Your rule-based assistant.",
                "You can call me DecodeBot, {name}.",
                "I'm your personal AI, built with if-else logic and dictionaries."
            ],
            "who are you": [
                "I'm a rule-based chatbot, {name}. No deep learning, just logic gates!",
                "I'm your deterministic conversation partner, {name}."
            ],

            # User questions about themselves (bot reflects)
            "what is my name": [
                "Your name is {name}, of course! Did you forget? 😄",
                "You told me your name is {name}. Am I right?"
            ],

            # Time (mock)
            "time": [
                "Sorry {name}, I don't have a real clock. Check your device!",
                "I'm just a rule-based bot, {name}. I cannot tell the time yet."
            ],
            "date": [
                "I don't track dates, {name}. But today is always a good day to code!"
            ],

            # Help
            "help": [
                "📋 *Available commands*, {name}:\n"
                "- Greetings: hello, hi, hey\n"
                "- Ask about me: what is your name, who are you\n"
                "- Ask about yourself: what is my name\n"
                "- Small talk: how are you, thanks, love, weather\n"
                "- Exit: bye, exit, quit, goodbye\n"
                "I also understand keywords inside longer sentences."
            ],

            # Thanks
            "thanks": [
                "You're welcome, {name}!",
                "Happy to help, {name}.",
                "Anytime, {name}!"
            ],
            "thank you": [
                "My pleasure, {name}.",
                "Glad to be useful, {name}."
            ],

            # General positive/negative
            "yes": [
                "Great, {name}!",
                "Awesome, {name}. Tell me more."
            ],
            "no": [
                "Oh, sorry to hear that, {name}.",
                "I see, {name}. Maybe I can help with something else."
            ],

            # Farewell – will be handled specially
            "bye": [
                "Goodbye {name}! See you next time.",
                "Take care, {name}! 👋",
                "Bye {name}. Come back soon!"
            ],
            "exit": [
                "Exiting now. Goodbye {name}!"
            ],
            "quit": [
                "Quitting. Have a nice day, {name}!"
            ],
            "goodbye": [
                "Goodbye, {name}. It was nice talking to you."
            ]
        }


        # ----- PART B: KEYWORD MATCH RESPONSES -----
        # These allow the bot to react to words inside longer sentences.
        # Keys are single keywords, values are lists of replies.
       self.keyword_responses = {
            "weather": [
                "I can't check real weather, {name}, but I hope it's sunny in your heart! ☀️"
            ],
            "rain": [
                "Don't forget your umbrella, {name}! ☔"
            ],
            "love": [
                "Love is a beautiful thing, {name}. I'm just code, but I appreciate the sentiment ❤️"
            ],
            "hate": [
                "Sorry you feel that way, {name}. Can I help with anything?"
            ],
            "python": [
                "Python is awesome, {name}! Do you want to learn more about dictionaries and loops?"
            ],
            "code": [
                "Coding is fun, {name}! This chatbot is built with pure Python logic."
            ],
            "robot": [
                "Yes, I am a robot, {name}. But a friendly one!"
            ],
            "smart": [
                "Thank you {name}! I'm just a set of rules, but I try my best."
            ],
            "dumb": [
                "Ouch, {name}. I'm doing my job with if-else statements."
            ],
            "food": [
                "I don't eat, {name}, but I hope you enjoy your next meal!"
            ],
            "fun": [
                "Glad you think so, {name}! Let's keep chatting."
            ]
        }

        # ----- PART C: EXIT WORDS (commands to stop the loop) -----
        # Using a set for O(1) membership test.
       self.exit_words = {"bye", "exit", "quit", "goodbye", "bye bye", "see you"}

        # ----- PART D: USER NAME STORAGE -----
        # Will be set when the bot asks the user at the beginning.
       self.user_name = None

   # SECTION 2.2: INPUT SANITISATION
   def sanitize(self, text):
        """
        Clean user input: lowercase, strip whitespace, remove punctuation.
        Returns a normalized string for exact matching.
        """
        text = text.lower().strip()
        # Remove punctuation characters
        text = text.translate(str.maketrans('', '', string.punctuation))
        # Remove extra spaces
        text = ' '.join(text.split())
        return text

   # SECTION 2.3: EXACT MATCH RESPONSE
   def get_exact_response(self, clean_input):
        """
        Check if the cleaned input exactly matches a key in the exact_responses dict.
        Returns a formatted reply (with {name} replaced) or None if no match.
        """
        if clean_input in self.exact_responses:
            replies = self.exact_responses[clean_input]
            reply = random.choice(replies)
            return reply.format(name=self.user_name)
        return None

   # SECTION 2.4: KEYWORD MATCH RESPONSE
   def get_keyword_response(self, clean_input):
        """
        Search for any keyword from keyword_responses inside the user's input.
        Returns first matching reply formatted with name.
        """
        for keyword, replies in self.keyword_responses.items():
            if keyword in clean_input:
                reply = random.choice(replies)
                return reply.format(name=self.user_name)
        return None

   # SECTION 2.5: FALLBACK RESPONSE (when no rule matches)
   def get_fallback_response(self):
        """
        Default response when no exact match and no keyword matches.
        """
        fallbacks = [
            f"Sorry {self.user_name}, I didn't understand that. Type 'help' to see what I can do.",
            f"Hmm {self.user_name}, I'm not programmed for that yet. Try rephrasing?",
            f"Could you explain differently, {self.user_name}? I'm a simple rule-based bot."
        ]
        return random.choice(fallbacks)

   # SECTION 2.6: MAIN RESPONSE ORCHESTRATOR
   def get_response(self, user_input):
        """
        Main orchestrator:
        1. Check for exit commands
        2. Try exact match
        3. Try keyword match
        4. Fallback
        """
        clean = self.sanitize(user_input)

        # Exit command handling
        if clean in self.exit_words or any(exit_w in clean for exit_w in self.exit_words):
            # Return a special token that signals the loop to break
            return "EXIT_TOKEN"

        # Exact match (fast, deterministic)
        exact_reply = self.get_exact_response(clean)
        if exact_reply:
            return exact_reply

        # Keyword-based match (flexible)
        keyword_reply = self.get_keyword_response(clean)
        if keyword_reply:
            return keyword_reply

        # Fallback
        return self.get_fallback_response()

   # SECTION 2.7: ASK FOR USER'S NAME (start of session)
   def ask_user_name(self):
        """
        Prompt the user for their name at the start of the session.
        Keep asking until a non-empty name is given.
        """
        print("\n🤖 Welcome to the DecodeLabs Rule-Based Chatbot! 🤖")
        print("I am a deterministic AI that replies using dictionaries and keywords.")
        while True:
            name_input = input("Please tell me your name: ").strip()
            if name_input:
                self.user_name = name_input
                print(f"\n✨ Nice to meet you, {self.user_name}! ✨")
                print("You can ask me things like 'hello', 'how are you', 'what is your name', 'time', or 'help'.")
                print("To end the conversation, type 'bye', 'exit', or 'quit'.\n")
                break
            else:
                print("I didn't catch that. Could you enter your name?")

   # SECTION 2.8: MAIN CHAT LOOP (the "heartbeat")
   def run(self):
        """
        Start the chatbot: ask for name, then enter the infinite chat loop.
        """
        self.ask_user_name()

        # Main conversation loop
        while True:
            user_input = input(f"{self.user_name}: ")
            if not user_input:
                continue   # ignore empty lines

            response = self.get_response(user_input)

            # Check for exit signal
            if response == "EXIT_TOKEN":
                # Pick a random goodbye message from the exact_responses that contains 'bye'
                goodbye_candidates = self.exact_responses.get("bye", ["Goodbye {name}!"] )
                farewell = random.choice(goodbye_candidates).format(name=self.user_name)
                print(f"Bot: {farewell}")
                print("👋 Session ended. Have a great day!")
                break

            print(f"Bot: {response}")

**SECTION 3: PROGRAM ENTRY POINT**

In [12]:
if __name__ == "__main__":
    bot = PersonalisedChatbot()   # create an instance of the chatbot
    bot.run()                     # start the conversation


🤖 Welcome to the DecodeLabs Rule-Based Chatbot! 🤖
I am a deterministic AI that replies using dictionaries and keywords.
Please tell me your name: Noor

✨ Nice to meet you, Noor! ✨
You can ask me things like 'hello', 'how are you', 'what is your name', 'time', or 'help'.
To end the conversation, type 'bye', 'exit', or 'quit'.

Noor: name
Bot: Could you explain differently, Noor? I'm a simple rule-based bot.
Noor: what is your name
Bot: You can call me DecodeBot, Noor.
Noor: nice
Bot: Could you explain differently, Noor? I'm a simple rule-based bot.
Noor: how are you
Bot: Doing well, Noor. And you?
Noor: fine
Bot: Could you explain differently, Noor? I'm a simple rule-based bot.
Noor: yes
Bot: Awesome, Noor. Tell me more.
Noor: time
Bot: Sorry Noor, I don't have a real clock. Check your device!
Noor: bye
Bot: Bye Noor. Come back soon!
👋 Session ended. Have a great day!
